<a href="https://colab.research.google.com/github/kuds/reinforce-tactics/blob/main/notebooks/ppo_bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Force headless pygame BEFORE anything imports pygame (directly or
# transitively). SDL picks its video driver at import time, so
# setting this later is a no-op for an already-loaded SDL backend.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")

# Reinforce Tactics — PPO Bootstrap Curriculum

Warm-start a `MaskablePPO` policy by walking it through a fixed curriculum of
`(map, opponent)` stages before handing the resulting checkpoint off to
self-play (`notebooks/ppo_training.ipynb`).

Six stages: starter map (random → simple → medium), then beginner map
(random → simple → medium). A stage advances only when its eval win-rate
stays at or above its `promotion_win_rate` for `patience` consecutive
evaluations. If a stage exhausts its `max_timesteps` budget without
promoting, the runner raises `CurriculumStalled` — fix the underlying
issue (reward shaping, hyperparams) before raising the budget.

Stage list lives in `configs/bootstrap.yaml`; edit there to tune
thresholds, budgets, or PPO hyperparams without touching this notebook.

**Runtime:** GPU recommended (L4 or better). CPU is possible but slower.

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q gymnasium stable-baselines3 sb3-contrib tensorboard pandas numpy torch matplotlib opencv-python-headless

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone repo and install as a package
import os
import sys
from pathlib import Path

REPO_DIR = Path("reinforce-tactics")
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
elif Path("notebooks").exists():
    # Already inside the repo
    os.chdir("..")
else:
    print("Cloning repository...")
    !git clone https://github.com/kuds/reinforce-tactics.git
    os.chdir(REPO_DIR)

# Install the package so all imports resolve
!pip install -q -e .

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Working directory: {os.getcwd()}")

## 2. Imports

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from sb3_contrib import MaskablePPO

from reinforcetactics.rl.bootstrap import (
    CurriculumStalled,
    load_bootstrap_config,
    run_curriculum,
)
from reinforcetactics.rl.evaluation import evaluate_model
from reinforcetactics.rl.masking import make_maskable_env

print("All imports successful.")

## 2b. Storage configuration

Choose where to save outputs. Set `USE_GOOGLE_DRIVE = True` to persist
results to Google Drive (recommended for Colab — files survive runtime
disconnects). Set `False` to use local/Colab storage.

Each execution saves outputs under a datetime-stamped subfolder
(e.g. `benchmarks/bootstrap/20250615_143022/`), so previous runs are
preserved automatically.

In [ ]:
USE_GOOGLE_DRIVE = True
DRIVE_BASE_DIR = "reinforce-tactics/benchmarks"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        SAVE_DIR = Path("/content/drive/MyDrive") / DRIVE_BASE_DIR
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        print("Google Drive mounted successfully.")
        print(f"Save directory: {SAVE_DIR}")
    except ImportError:
        print("WARNING: google.colab not available (not running in Colab).")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = None
    except Exception as e:
        print(f"WARNING: Failed to mount Google Drive: {e}")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = None
else:
    SAVE_DIR = None
    print("Using local storage (default).")
    print("  Tip: Set USE_GOOGLE_DRIVE = True to persist results to Google Drive.")

## 3. Configuration

Reads `configs/bootstrap.yaml`. To experiment, copy that file, edit, and
point `CONFIG_PATH` at the copy.

Each stage can override `max_steps`, `max_turns`, and `ent_coef` to
handle map-size or exploration shifts (the bigger `beginner.csv` map
needs a longer turn budget; the first beginner stage gets an entropy
bump to break the starter-map policy out of its rut). The summary
table below highlights any active overrides per stage; absent values
inherit `cfg.env` / `cfg.ppo` defaults.

In [ ]:
CONFIG_PATH = Path("configs/bootstrap.yaml")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
if SAVE_DIR is not None:
    OUTPUT_DIR = SAVE_DIR / "bootstrap" / RUN_ID
else:
    OUTPUT_DIR = Path("benchmarks") / "bootstrap" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = load_bootstrap_config(CONFIG_PATH)

print(f"Config:        {CONFIG_PATH}")
print(f"Output dir:    {OUTPUT_DIR}")
print(f"Defaults:      max_steps={cfg.env.max_steps}, max_turns={cfg.env.max_turns}, ent_coef={cfg.ppo.ent_coef}")
print(f"Stages:        {len(cfg.stages)}")

# Per-stage table -- highlight per-stage overrides so the curriculum's
# map-transition handling (longer turn budget, entropy bump, reward
# reshape) is visible at a glance.
header = (
    f"  {'name':26s}  {'opp':16s}  {'WR>=':>5s}  "
    f"{'budget':>10s}  {'max_steps':>10s}  {'max_turns':>10s}  "
    f"{'ent_coef':>13s}  {'reward keys overridden'}"
)
print(header)
print("  " + "-" * (len(header) - 2))


def _fmt_override(value, default):
    if value is None:
        return f"({default})"
    # ``ent_coef`` may be a {start, end, schedule} mapping -- render
    # it compactly as e.g. ``0.10->0.03*`` so the table stays one line.
    if isinstance(value, dict):
        return f"{value.get('start', '?')}->{value.get('end', '?')}*"
    return f"{value}*"


for s in cfg.stages:
    reward_keys = sorted((s.reward_config or {}).keys())
    reward_str = ", ".join(reward_keys) if reward_keys else "(default)"
    print(
        f"  {s.name:26s}  {s.opponent:16s}  "
        f"{s.promotion_win_rate:>4.0%}  {s.max_timesteps:>10,}  "
        f"{_fmt_override(s.max_steps, cfg.env.max_steps):>10s}  "
        f"{_fmt_override(s.max_turns, cfg.env.max_turns):>10s}  "
        f"{_fmt_override(s.ent_coef, cfg.ppo.ent_coef):>13s}  "
        f"{reward_str}"
    )
print("  (* = per-stage override; (n) = inherits cfg default)")

print(f"\nEval freq:     {cfg.eval_freq:,} env steps")
print(f"n_envs:        {cfg.n_envs}")
print(f"Total budget:  {sum(s.max_timesteps for s in cfg.stages):,} env steps (worst case)")
print(f"Storage:       {'Google Drive' if USE_GOOGLE_DRIVE else 'Local'}")

## 4. Run the curriculum

One `model.learn()` call per stage, with the env swapped between stages
via `model.set_env(...)`. The shared `MaskablePPO` instance survives the
whole run, so the policy carries forward; only the `(map, opponent)` pair
changes. Best-by-eval-win-rate checkpoints land in
`OUTPUT_DIR/<stage_name>/best_model.zip`; the post-promotion model is
written as `stage_final.zip`.

In [ ]:
try:
    result = run_curriculum(cfg, output_dir=OUTPUT_DIR)
    stalled = None
except CurriculumStalled as exc:
    print(f"\nSTALLED: {exc}")
    print("Investigate before raising the budget — a stuck stage usually")
    print("signals reward shaping or hyperparam issues, not insufficient steps.")
    # Recover the partial run -- history through the stalled stage,
    # plus a final_model.zip saved at the moment of stall -- so the
    # rest of the notebook (diagnostics, checkpoint snapshot, replay
    # videos) still runs end-to-end. Without this, a stall would
    # discard every artifact you'd want for triage.
    result = exc.partial_result()
    stalled = exc

if result is not None:
    if result.get("stalled"):
        print(f"\nPartial run available (stalled at '{result['stalled_stage']}'):")
        if result.get("final_model_path"):
            print(f"Snapshot at stall: {result['final_model_path']}")
    else:
        print(f"\nFinal model: {result['final_model_path']}")
    for h in result["history"]:
        last = h["results"][-1] if h["results"] else None
        last_wr = f"{last['win_rate']:.1%}" if last else "n/a"
        print(f"  {h['stage']:18s}  promoted={h['promoted']!s:5s}  best_wr={h['best_win_rate']:.1%}  last_wr={last_wr}")

## 4b. Snapshot per-stage checkpoints

`run_curriculum` writes three files into `OUTPUT_DIR/<stage_name>/`
for each stage as soon as the stage finishes — `stage_final.zip`
(post-promotion model), `best_model.zip` (best by eval win-rate
during the stage), and `config.json` (resolved env + reward + PPO
hyperparams + seed + git/library/hardware metadata). Writing
`config.json` inside the runner means a Colab disconnect or a later
`CurriculumStalled` on a subsequent stage still leaves every
completed stage with a reproducible config snapshot on disk.

This cell mirrors the two checkpoint zips into a flat
`OUTPUT_DIR/checkpoints/` directory so every stage's policy is easy
to reload later — useful for replaying a specific (map, opponent)
combination or comparing intermediate policies side-by-side without
walking the per-stage subfolders.

In [ ]:
import shutil

if result is None:
    stage_checkpoints = {}
    print("Skipped — no successful run to snapshot.")
else:
    checkpoints_dir = OUTPUT_DIR / "checkpoints"
    checkpoints_dir.mkdir(parents=True, exist_ok=True)

    # One row per stage with the canonical paths the rest of the
    # notebook (sanity eval, replay video) consumes. Prefer
    # ``best_model.zip`` when it's strictly better than the post-stage
    # snapshot — it's the highest eval-WR checkpoint within the stage.
    # ``run_curriculum`` already wrote ``config.json`` next to each
    # stage's checkpoints; we don't re-emit it here.
    stage_checkpoints = {}
    for h in result["history"]:
        stage_name = h["stage"]
        stage_dir = OUTPUT_DIR / stage_name
        stage_final = stage_dir / "stage_final.zip"
        best_model = stage_dir / "best_model.zip"

        # Copy stage_final under the stage's name; copy best_model with
        # a `_best` suffix when present. Both stay paired with the
        # source folder so the per-stage subdirs remain authoritative.
        flat_final = checkpoints_dir / f"{stage_name}.zip"
        if stage_final.exists():
            shutil.copy2(stage_final, flat_final)
        flat_best = checkpoints_dir / f"{stage_name}_best.zip"
        if best_model.exists():
            shutil.copy2(best_model, flat_best)

        stage_checkpoints[stage_name] = {
            "map_file": next(s.map_file for s in cfg.stages if s.name == stage_name),
            "opponent": next(s.opponent for s in cfg.stages if s.name == stage_name),
            "stage_final": str(flat_final) if flat_final.exists() else None,
            "best_model": str(flat_best) if flat_best.exists() else None,
            "promoted": h["promoted"],
            "best_win_rate": h["best_win_rate"],
        }

    print(f"Snapshotted {len(stage_checkpoints)} stage checkpoints to:")
    print(f"  {checkpoints_dir}")
    print()
    for name, meta in stage_checkpoints.items():
        promo = "promoted" if meta["promoted"] else "stalled"
        final_path = Path(meta["stage_final"]).name if meta["stage_final"] else "(missing)"
        best_path = Path(meta["best_model"]).name if meta["best_model"] else "(no best)"
        print(f"  {name:30s}  {promo:8s}  best_wr={meta['best_win_rate']:5.1%}  final={final_path}  best={best_path}")

## 5. Win-rate over the full curriculum

Concatenates eval snapshots across stages onto a single timestep axis.
Vertical lines mark stage transitions; horizontal lines mark each stage's
promotion threshold. A regression bump right after a transition is
expected with hard cutoffs — the policy is encountering a new opponent
for the first time.

In [ ]:
if result is None:
    print("Skipped \u2014 no successful run to plot.")
else:
    fig, ax = plt.subplots(figsize=(12, 5))
    cmap = plt.get_cmap("tab10")
    for i, h in enumerate(result["history"]):
        if not h["results"]:
            continue
        xs = [r["timesteps"] for r in h["results"]]
        ys = [r["win_rate"] for r in h["results"]]
        ax.plot(xs, ys, marker="o", label=h["stage"], color=cmap(i % 10))
        # Promotion threshold for the stage.
        stage_cfg = next(s for s in cfg.stages if s.name == h["stage"])
        ax.hlines(
            stage_cfg.promotion_win_rate,
            xmin=xs[0],
            xmax=xs[-1],
            colors=cmap(i % 10),
            linestyles=":",
            alpha=0.5,
        )
        # Stage transition marker at the last timestep of the stage.
        ax.axvline(xs[-1], color="gray", linestyle="--", alpha=0.3)
    ax.set_xlabel("Cumulative env timesteps")
    ax.set_ylabel("Eval win rate")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title("Curriculum win rate (dotted = stage threshold, dashed = transition)")
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=9)
    out = OUTPUT_DIR / "curriculum_winrate.png"
    fig.tight_layout()
    fig.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")

## 5b. Stage diagnostics: outcomes / length / reward mix / action mix

Four panels per stage (drawn separately so each stage's timestep axis
is locally meaningful):

1. **W/L/D over time** — `0/0/30` (all draws) means episodes are timing
   out without a winner; `0/30/0` (all losses) means the opponent is
   crushing the agent. Different fixes for each.
2. **Mean episode length** — saturating at `max_steps` confirms the
   turn cap is the rate-limiter; climbing then dropping is the policy
   learning to win earlier.
3. **Reward decomposition** — large green `action` band with ~0
   `terminal` band reads as 'farms shaping but never wins'. Growing
   orange `terminal` band is healthy convergence.
4. **Action mix** — dominant `end_turn` is policy collapse;
   `create_unit`-heavy with no `attack`/`seize` is over-economy.

In [ ]:
from reinforcetactics.rl.viz import (
    plot_action_distribution,
    plot_combat_summary,
    plot_outcome_breakdown,
    plot_reward_decomposition,
    plot_unit_build_distribution,
)

if result is None:
    print("Skipped — no successful run to diagnose.")
else:
    diagnostics_dir = OUTPUT_DIR / "diagnostics"
    diagnostics_dir.mkdir(parents=True, exist_ok=True)

    for h in result["history"]:
        if not h["results"]:
            print(f"[{h['stage']}] no eval snapshots, skipping diagnostics")
            continue
        stage_charts_dir = diagnostics_dir / h["stage"]
        stage_charts_dir.mkdir(parents=True, exist_ok=True)

        # Mean episode length per eval -- a small standalone chart so
        # the saturating-at-cap pattern is unmistakable.
        results_seq = h["results"]
        timesteps = [r["timesteps"] for r in results_seq]
        lengths = [r["avg_length"] for r in results_seq]
        fig_len, ax_len = plt.subplots(figsize=(10, 3.2))
        ax_len.plot(timesteps, lengths, marker="o")
        ax_len.set_xlabel("Timesteps")
        ax_len.set_ylabel("Mean episode length")
        ax_len.set_title(f"[{h['stage']}] Episode length per eval")
        ax_len.grid(alpha=0.3)
        fig_len.tight_layout()
        fig_len.savefig(stage_charts_dir / "episode_length.png", dpi=120, bbox_inches="tight")
        plt.show()

        # The viz helpers expect ``.results``-shaped lists and write
        # their png next to whatever ``charts_dir`` we pass.
        plot_outcome_breakdown(results_seq, charts_dir=stage_charts_dir)
        plt.show()
        plot_reward_decomposition(results_seq, charts_dir=stage_charts_dir)
        plt.show()
        plot_action_distribution(results_seq, charts_dir=stage_charts_dir)
        plt.show()
        # Build mix and combat counters: surfaces "what is the agent
        # actually doing" beyond the raw action distribution -- which
        # units it's making, whether attacks are landing damage, and
        # whether it's getting hit back.
        plot_unit_build_distribution(results_seq, charts_dir=stage_charts_dir)
        plt.show()
        plot_combat_summary(results_seq, charts_dir=stage_charts_dir)
        plt.show()
        print(f"[{h['stage']}] diagnostics saved to {stage_charts_dir}")

## 5c. PPO update health across the curriculum

Six panels on a shared timestep axis spanning the entire bootstrap
run: top row is the per-eval scalars (win rate, avg reward, episode
length); bottom row is the per-rollout PPO internals (`approx_kl`,
`explained_variance`, `value_loss`). Vertical dashed lines mark
stage transitions so you can see whether the value head holds up
across map / opponent shifts or has to re-fit from scratch.

**What to look for:**

- **`approx_kl` near zero across a stage** — policy isn't actually
  updating. Usually means the gradient is being absorbed by value-head
  fitting; check the next two panels.
- **`explained_variance` flat near zero or negative** — value head
  isn't fitting the returns. With reward magnitudes in the thousands
  this is the most likely failure mode of large-scale shaping.
- **`value_loss` growing without bound** (especially on log scale) —
  value targets are outpacing the head's capacity / lr. Consider
  scaling all reward weights down by 10–100x or adding `clip_range_vf`.
- **Sharp `value_loss` spike at a stage transition that decays back
  down** — healthy: the value head is re-fitting the new reward scale.
  A spike that *stays* elevated is the diagnosis pattern for the
  "catastrophic value dislocation" mentioned in the bootstrap config
  comments.

In [ ]:
from reinforcetactics.rl.viz import plot_eval_curves

if result is None or not result.get("history"):
    print("Skipped — no eval history.")
else:
    metrics_callback = result.get("metrics_callback")
    train_records = metrics_callback.records if metrics_callback is not None else None

    # Concatenate per-stage eval results onto the cumulative timestep
    # axis. ``plot_eval_curves`` expects a single flat list; the
    # per-stage results already carry cumulative timesteps because the
    # runner uses ``reset_num_timesteps=False``.
    all_results = []
    for h in result["history"]:
        all_results.extend(h["results"])

    # Stage-transition x-coordinates: take the last eval timestep of
    # each promoted stage (excluding the final stage, which doesn't
    # transition to anything). Vertical dashed lines on every panel.
    stage_boundaries = []
    for h in result["history"][:-1]:
        if h["results"]:
            stage_boundaries.append(h["results"][-1]["timesteps"])

    if not all_results:
        print("Skipped — no eval results across stages.")
    else:
        plot_eval_curves(
            all_results,
            train_records=train_records,
            opponent_label="curriculum",
            charts_dir=OUTPUT_DIR,
            stage_boundaries=stage_boundaries,
        )
        plt.show()

## 6. Sanity eval on the final stage

Reload the saved checkpoint and run a fresh evaluation against the
hardest stage (beginner map vs medium bot) to confirm the exit criterion
really holds. Uses more episodes than the in-training eval for a tighter
estimate.

In [ ]:
if result is None or not result.get("history") or not result.get("final_model_path"):
    print("Skipped — no final checkpoint to sanity-eval.")
else:
    # On a successful run this is ``cfg.stages[-1]``. On a stall we
    # never reached the configured final stage, so eval against the
    # last stage we *did* train on -- the same one the saved
    # ``final_model.zip`` was last optimised against -- so the
    # numbers are interpretable.
    last_stage_name = result["history"][-1]["stage"]
    final_stage = next(s for s in cfg.stages if s.name == last_stage_name)
    sanity_env = make_maskable_env(
        map_file=final_stage.map_file,
        opponent=final_stage.opponent,
        max_steps=final_stage.resolve_max_steps(cfg.env),
        max_turns=final_stage.resolve_max_turns(cfg.env),
        reward_config=final_stage.resolve_reward_config(cfg.env),
        enabled_units=cfg.env.enabled_units,
        action_space_type=cfg.env.action_space_type,
        seed=cfg.seed + 9999,  # different seed than training eval
        opponent_kwargs=final_stage.opponent_kwargs,
    )
    loaded = MaskablePPO.load(result["final_model_path"])
    metrics = evaluate_model(loaded, sanity_env, n_episodes=100, seed=cfg.seed + 9999)
    sanity_env.close()
    label = f"{final_stage.name}{' [stalled]' if result.get('stalled') else ''}"
    print(
        f"Sanity eval ({label}, n=100):  "
        f"WR={metrics['win_rate']:.1%}  "
        f"reward={metrics['avg_reward']:+.1f}  "
        f"W/L/D={metrics['wins']}/{metrics['losses']}/{metrics['draws']}"
    )
    if metrics["win_rate"] < final_stage.promotion_win_rate:
        print(
            f"  WARNING: sanity WR {metrics['win_rate']:.1%} < threshold "
            f"{final_stage.promotion_win_rate:.1%}. The policy may be\n"
            "  overfit to the training eval seed. Consider a longer final\n"
            "  stage or rolling-window promotion before warm-starting self-play."
        )

## 7. Hand off to self-play

The final checkpoint is the artifact `notebooks/ppo_training.ipynb`'s
self-play setup should load. Drop this snippet into that notebook in
place of the fresh `MaskablePPO(...)` constructor call:

```python
from sb3_contrib import MaskablePPO

BOOTSTRAP_CHECKPOINT = '<paste path printed below>'
model = MaskablePPO.load(BOOTSTRAP_CHECKPOINT, env=self_play_vec_env)
```

`set_env()` is called implicitly by `MaskablePPO.load(..., env=...)`, so
the policy weights load while the env is the new self-play one.

In [ ]:
if result is None or not result.get("final_model_path"):
    print("No checkpoint to hand off.")
elif result.get("stalled"):
    print(f"PARTIAL hand-off only — curriculum stalled at '{result['stalled_stage']}'.")
    print("This checkpoint was NOT trained against the configured final stage,")
    print("so don't warm-start self-play with it without first fixing the stall and re-running.")
    print(f"  {result['final_model_path']}")
else:
    print("Hand-off path:")
    print(f"  {result['final_model_path']}")
    print("\nIn ppo_training.ipynb:")
    print(f"  BOOTSTRAP_CHECKPOINT = '{result['final_model_path']}'")
    print("  model = MaskablePPO.load(BOOTSTRAP_CHECKPOINT, env=self_play_vec_env)")

## 8. Replay videos per (map, opponent)

Loads each stage's snapshotted checkpoint (preferring the best-model
copy when present) and records one full episode against that stage's
opponent on that stage's map. Output goes to `OUTPUT_DIR/videos/` as
`<stage_name>.mp4` plus a paired `.json` replay. Useful for eyeballing
*what* the policy actually learned at each curriculum step — win-rate
curves tell you whether it works, the videos tell you how it works.

Skips stages whose checkpoint is missing. If `imageio_ffmpeg` isn't
installed the writer falls back to OpenCV's `mp4v` codec (may not
play in QuickTime / Preview but is fine in browser previews).

In [ ]:
from reinforcetactics.utils.video import record_evaluation_to_video

if result is None or not stage_checkpoints:
    print("Skipped — no checkpoints to replay.")
    video_summary = []
else:
    videos_dir = OUTPUT_DIR / "videos"
    videos_dir.mkdir(parents=True, exist_ok=True)

    video_summary = []
    for stage_name, meta in stage_checkpoints.items():
        # Prefer the best-by-WR snapshot when available; the final
        # post-promotion checkpoint is the fallback. A stage with a
        # missing zip (rare — write failure mid-run) is skipped.
        ckpt_path = meta["best_model"] or meta["stage_final"]
        if ckpt_path is None:
            print(f"[{stage_name}] no checkpoint, skipping video")
            continue

        # Match the stage's training env so the replay is faithful to
        # what the policy was optimised on (max_steps / max_turns /
        # reward_config can shift between stages).
        stage_cfg = next(s for s in cfg.stages if s.name == stage_name)
        replay_env = make_maskable_env(
            map_file=stage_cfg.map_file,
            opponent=stage_cfg.opponent,
            max_steps=stage_cfg.resolve_max_steps(cfg.env),
            max_turns=stage_cfg.resolve_max_turns(cfg.env),
            reward_config=stage_cfg.resolve_reward_config(cfg.env),
            enabled_units=cfg.env.enabled_units,
            action_space_type=cfg.env.action_space_type,
            seed=cfg.seed + 12345,
            opponent_kwargs=stage_cfg.opponent_kwargs,
        )
        loaded = MaskablePPO.load(ckpt_path)

        video_path = videos_dir / f"{stage_name}.mp4"
        try:
            info = record_evaluation_to_video(
                replay_env,
                loaded,
                output_path=str(video_path),
                fps=4,
                max_steps=stage_cfg.resolve_max_steps(cfg.env),
                deterministic=True,
                scale=4,
                use_pixel_art=True,
            )
            video_summary.append(
                {
                    "stage": stage_name,
                    "map_file": meta["map_file"],
                    "opponent": meta["opponent"],
                    "video_path": info.get("video_path"),
                    "winner": info.get("winner"),
                    "end_reason": info.get("end_reason"),
                    "steps": info.get("steps"),
                    "total_reward": info.get("total_reward"),
                    # Per-step traces from record_evaluation_to_video --
                    # required by section 9 (individual game stats 2x3).
                    # Each entry has unit/gold/structure counts plus the
                    # per-step action_type / reward_breakdown so the
                    # downstream plotter can build cumulative reward
                    # curves and an action-mix bar without re-running
                    # the recording.
                    "step_stats": info.get("step_stats", []),
                }
            )
            print(
                f"[{stage_name}] map={meta['map_file']}  opp={meta['opponent']}  "
                f"winner={info.get('winner')}  end={info.get('end_reason')}  "
                f"steps={info.get('steps')}  -> {info.get('video_path')}"
            )
        except Exception as exc:
            # Don't let a single rendering hiccup take out the rest of
            # the loop -- ffmpeg / pygame failures are usually
            # environment-specific.
            print(f"[{stage_name}] video failed: {exc}")
        finally:
            replay_env.close()

    print()
    print(f"Saved {len(video_summary)} replay video(s) to {videos_dir}")

## 9. Per-stage individual game statistics (2x3)

For each replay recorded above, plot a 2x3 panel of per-step
diagnostics. Useful for asking "what did the agent *actually* do this
game?" — independent of the aggregate eval curves in section 5/5b.

- **Top row (per player over time):** unit count, gold, structures
  owned (towers + buildings + HQ).
- **Bottom-left:** action mix used in this game — `create_unit` is
  split by which type was actually spawned (`create_W`, `create_M`, …)
  so you can see e.g. "the agent only ever built Warriors". Blue bars
  = creates, green bars = other actions.
- **Bottom-middle:** cumulative reward by component (`action`,
  `shaping_delta`, `invalid_penalty`, `terminal`).
- **Bottom-right:** outcome summary — winner, end reason
  (`hq_capture` / `elimination` / `max_turns_draw` /
  `max_steps_truncate`), final unit/gold/structure counts, and
  per-type creation counts.

Each panel reads from the `step_stats` returned by
`record_evaluation_to_video`. One file per stage lands at
`OUTPUT_DIR/individual_game_stats/<stage_name>.png`.

In [ ]:
from reinforcetactics.rl.viz import plot_individual_game_stats

if not video_summary:
    print("Skipped — no recorded games to plot. Run section 8 first.")
else:
    stats_dir = OUTPUT_DIR / "individual_game_stats"
    stats_dir.mkdir(parents=True, exist_ok=True)
    for entry in video_summary:
        if not entry.get("step_stats"):
            print(f"[{entry['stage']}] no step_stats on entry, skipping")
            continue
        # ``filename`` keeps the bootstrap output flat: one file per
        # stage in stats_dir, named ``<stage>.png``. The library helper
        # defaults to ``individual_game_stats[_<title_suffix>].png``
        # (matching ppo_training.ipynb 9e); we override here because
        # the directory name already namespaces these.
        plot_individual_game_stats(
            entry,
            charts_dir=stats_dir,
            title_suffix=entry["stage"],
            filename=f"{entry['stage']}.png",
        )
        plt.show()
    print(f"\nSaved per-stage stats panels to {stats_dir}")

In [ ]:
import time

print("Training finished. Disconnecting runtime in 5 seconds...")
time.sleep(5)

from google.colab import runtime

runtime.unassign()